In [1]:
from pathlib import Path
import json, sys, hashlib
import numpy as np
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AGENTS.md').exists())
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data/research/cryptoquant_text_20261023'
result = json.loads((DATA / 'results.json').read_text(encoding='utf-8'))
print(result['quality'])

{'input_rows': 2517, 'usable': 2339, 'edited_excluded': 156, 'duplicate_text': 22, 'unique_source_ids': 2517, 'labelled': 2339, 'label_classes': {'descriptive': 1304, 'promotion': 152, 'directional': 466, 'other': 149, 'conditional': 252, 'technical': 16}, 'eligibility_rejections': {'no_explicit_allowed_asset': 141, 'events_on_nondirectional_category': 83}, 'first_publication': '2021-05-19T13:19:22+00:00', 'last_publication': '2026-10-23T13:04:58+00:00', 'labels_sha256': 'fdee194dbdb75d36b3423afc3afd42360b0a52a610e73f5b58ad2e0253074cb9', 'labels_frozen_before_price_join': True}


In [2]:
for relative, meta in result['market_sources'].items():
    snapshot = DATA / 'market_snapshot' / relative
    assert hashlib.sha256(snapshot.read_bytes()).hexdigest() == meta['sha256']
assert hashlib.sha256((DATA / 'frozen_labels.jsonl').read_bytes()).hexdigest() == result['quality']['labels_sha256']
print('All text-label and market-snapshot hashes match')

All text-label and market-snapshot hashes match


In [3]:
from scripts.research_telegram_text import Market, independent_events
from core.research.telegram_text_signals import sha
labels = [json.loads(line) for line in (DATA / 'frozen_labels.jsonl').read_text(encoding='utf-8').splitlines()]
reviewed = []
for label in labels:
    events = []
    for event in label['events']:
        if event['confidence'] < .7:
            continue
        review = json.loads((DATA / 'semantic_review' / (sha(label['source_id'] + '|' + event['symbol']) + '.json')).read_text(encoding='utf-8'))
        if review['accept']:
            events.append(event)
    reviewed.append({**label, 'events': events})
market = Market(DATA / 'market_snapshot')
for selection, inputs in [('raw', labels), ('semantic_reviewed', reviewed)]:
    entries, skipped = independent_events(inputs, 'cryptoquant_official', 'BTCUSDT', 24, 'holdout')
    net = [market.outcome('BTCUSDT', r['entry_at'], 24, r['side']).get('net') for r in entries]
    net = [v for v in net if v is not None]
    recorded = next(r for r in result['results'] if r['signal_set'] == selection and r['symbol'] == 'BTCUSDT' and r['phase'] == 'holdout' and r['hours'] == 24)['funded']
    assert len(net) == recorded['n']
    assert abs(np.mean(net) - recorded['mean']) < 1e-12
    print(selection, len(net), 'mean net', round(np.mean(net)*100, 4), '%')

raw 41 mean net 0.3614 %
semantic_reviewed 28 mean net 0.5972 %
